In [ ]:
# LSTM classifier for nurse stress prediction from sliding windows.
# If needed, install dependencies in a terminal: pip install tensorflow scikit-learn pandas numpy matplotlib
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

## 1) Configuration and Helper Functions

This section defines windowing rules and preprocessing helpers for sequence modeling.

In [ ]:
DATA_DIR = Path('data/Eric')
WINDOW_SECONDS = 10.0
FIXED_WINDOW_STEPS = None  # Set an integer (e.g., 256) to force a fixed sequence length.
MAX_WINDOW_STEPS = 300      # Safety cap to avoid very large tensors.
STRIDE_FRACTION = 0.50
TEST_RATIO = 0.20

FEATURE_COLS = ['acc_mag', 'EDA', 'HR', 'TEMP']
TARGET_COL = 'label'

def infer_window_steps(time_values: np.ndarray, window_seconds: float) -> int:
    if len(time_values) < 2:
        return 1
    diffs = np.diff(time_values)
    positive_diffs = diffs[diffs > 0]
    if len(positive_diffs) == 0:
        return 1
    dt = float(np.median(positive_diffs))
    return max(1, int(round(window_seconds / dt)))

def build_sequences(
    X: np.ndarray,
    y: np.ndarray,
    window_steps: int,
    stride_steps: int,
    end_point_label: bool = True,
 ) -> tuple[np.ndarray, np.ndarray]:
    if len(X) < window_steps:
        return np.empty((0, window_steps, X.shape[1]), dtype=np.float32), np.empty((0,), dtype=np.int8)

    X_seq = []
    y_seq = []
    for start in range(0, len(X) - window_steps + 1, stride_steps):
        end = start + window_steps
        X_seq.append(X[start:end])
        if end_point_label:
            y_seq.append(int(y[end - 1]))
        else:
            y_seq.append(int(np.round(y[start:end].mean())))

    return np.asarray(X_seq, dtype=np.float32), np.asarray(y_seq, dtype=np.int8)

def scale_3d(train_3d: np.ndarray, test_3d: np.ndarray) -> tuple[np.ndarray, np.ndarray, StandardScaler]:
    scaler = StandardScaler()
    n_features = train_3d.shape[-1]
    train_2d = train_3d.reshape(-1, n_features)
    test_2d = test_3d.reshape(-1, n_features)
    train_scaled = scaler.fit_transform(train_2d).reshape(train_3d.shape).astype(np.float32)
    test_scaled = scaler.transform(test_2d).reshape(test_3d.shape).astype(np.float32)
    return train_scaled, test_scaled, scaler

## 2) Load Data and Build LSTM Sequences

Creates rolling windows per nurse and concatenates all windows for model training.

In [ ]:
csv_files = sorted(DATA_DIR.glob('processed_nurse_*.csv'))
if not csv_files:
    raise FileNotFoundError(f'No processed_nurse_*.csv files found in {DATA_DIR}')

X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []
nurse_stats = []

for csv_path in csv_files:
    df = pd.read_csv(csv_path)
    required = set(FEATURE_COLS + [TARGET_COL, 'time'])
    missing = required - set(df.columns)
    if missing:
        print(f'Skipping {csv_path.name}: missing columns {sorted(missing)}')
        continue

    df = df.sort_values('time').reset_index(drop=True)
    X = df[FEATURE_COLS].to_numpy(dtype=np.float32)
    y = (df[TARGET_COL] > 0).to_numpy(dtype=np.int8)

    window_steps = FIXED_WINDOW_STEPS if FIXED_WINDOW_STEPS is not None else infer_window_steps(df['time'].to_numpy(dtype=np.float32), WINDOW_SECONDS)
    window_steps = int(max(1, min(window_steps, MAX_WINDOW_STEPS)))
    stride_steps = int(max(1, round(window_steps * STRIDE_FRACTION)))

    X_seq, y_seq = build_sequences(X, y, window_steps=window_steps, stride_steps=stride_steps, end_point_label=True)
    if len(X_seq) < 2:
        print(f'Skipping {csv_path.name}: too few sequences ({len(X_seq)})')
        continue

    split_idx = int(len(X_seq) * (1.0 - TEST_RATIO))
    split_idx = min(max(split_idx, 1), len(X_seq) - 1)

    X_train_list.append(X_seq[:split_idx])
    y_train_list.append(y_seq[:split_idx])
    X_test_list.append(X_seq[split_idx:])
    y_test_list.append(y_seq[split_idx:])

    nurse_stats.append({
        'nurse_id': csv_path.stem.replace('processed_nurse_', ''),
        'window_steps': window_steps,
        'stride_steps': stride_steps,
        'num_sequences': int(len(X_seq)),
    })

if not X_train_list:
    raise RuntimeError('No valid nurse sequences were created. Check your data and settings.')

# LSTM needs a fixed sequence length across all samples, so keep the common minimum length.
common_steps = min(arr.shape[1] for arr in X_train_list + X_test_list)
X_train = np.concatenate([arr[:, :common_steps, :] for arr in X_train_list], axis=0)
y_train = np.concatenate(y_train_list, axis=0)
X_test = np.concatenate([arr[:, :common_steps, :] for arr in X_test_list], axis=0)
y_test = np.concatenate(y_test_list, axis=0)

X_train, X_test, scaler = scale_3d(X_train, X_test)

print('Nurses used:', len(nurse_stats))
print('X_train shape:', X_train.shape, 'y_train shape:', y_train.shape)
print('X_test shape:', X_test.shape, 'y_test shape:', y_test.shape)
pd.DataFrame(nurse_stats).head()

## 3) Define and Train LSTM

A compact LSTM architecture with early stopping to reduce overfitting.

In [ ]:
n_timesteps = X_train.shape[1]
n_features = X_train.shape[2]

model = Sequential([
    Input(shape=(n_timesteps, n_features)),
    LSTM(64, return_sequences=True),
    Dropout(0.25),
    LSTM(32),
    Dropout(0.20),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')],
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5),
]

history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1,
 )

model.summary()

## 4) Evaluate and Visualize

Computes test metrics and plots training/validation curves.

In [ ]:
test_metrics = model.evaluate(X_test, y_test, verbose=0)
print(dict(zip(model.metrics_names, test_metrics)))

y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(np.int8)

print('Test Accuracy:', accuracy_score(y_test, y_pred))
print('Test F1:', f1_score(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred, digits=4))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.show()